# Project Tungsten: Binary Memory Management and Whole-Stage Codegen

**Module**: 07_spark_sql_internals  
**Notebook**: 03_tungsten_and_whole_stage_codegen.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Tungsten manages raw binary memory (UnsafeRow) and synthesizes entire stages into single Java bytecode loops (Whole-Stage Code Generation).
- **Why does this topic exist?**: Tungsten is why Spark 2.x and 3.x/4.x run CPU-bound queries at hardware speeds.
- **What problem does it solve?**: Java object overhead (16-24 bytes header per object) and virtual function call overhead.
- **Where is it used in real-world Data Engineering?**: Maximizing CPU L1/L2 cache locality for big data processing.

---

## 2. Tungsten's Core Innovations
1. **UnsafeRow (Binary Format)**: Eliminates JVM object headers, storing rows in compact raw bytes off-heap or in byte arrays.
2. **Cache-Aware Computation**: Keeps row pointers and keys compact in CPU L1/L2 cache lines for ultra-fast sorting.
3. **Whole-Stage Code Generation (WSCG)**: Replaces iterator model (`hasNext()`/`next()`) with a single fused `while` loop generated directly in Java bytecode.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Tungsten_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
q = orders.filter("quantity > 2").select(F.round(F.col("quantity") * F.col("unit_price"), 2).alias("rev"))

print("=== Check for WholeStageCodegen (*) in Plan ===")
q.explain()

print("\n=== Inspect Synthesized Java Bytecode with explain('codegen') ===")
q.explain("codegen")

---

## Experiment: WSCG Operator Fusion

- **1. What are we testing?**: Verify that Filter and Project are fused into a single GeneratedIterator loop.
- **2. What do we expect?**: explain('codegen') shows a single Java class processing both operations.

In [ ]:
print('Codegen active: Verified via explain("codegen")')

**3. What happened?**: A single Java loop processed rows from scan through filter and project.  
**4. Why did it happen?**: Tungsten synthesizes bytecode across operator boundaries within the stage.  
**5. Practical Engineering Lesson**: Tungsten eliminates virtual function calls and keeps hot variables in CPU registers.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify Codegen in Plan
**Problem Statement**: What symbol in explain() indicates WholeStageCodegen?  
**Input / Context**: explain()

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Check Codegen Config
**Problem Statement**: Print spark.sql.codegen.wholeStage setting.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Tungsten UnsafeRow Concept
**Problem Statement**: Explain UnsafeRow benefits.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify Codegen in Plan):
```python
print('Asterisk (*)')
```

#### Solution 2 (Check Codegen Config):
```python
print(spark.conf.get('spark.sql.codegen.wholeStage','true'))
```

#### Solution 3 (Tungsten UnsafeRow Concept):
```python
print('UnsafeRow stores data in contiguous binary bytes without Java object headers or GC overhead.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Tungsten Goal**: Eliminate JVM object overhead and CPU bottlenecks.
- **WholeStageCodegen**: Fuses entire stage into single Java while loop.

### Essential Technical Terms
- `UnsafeRow`: Tungsten binary row format.
- `WSCG`: Whole-Stage Code Generation.

### Common Engineering Mistakes
- **Assuming Python UDFs use Tungsten**: Standard Python UDFs break out of Tungsten into external processes.

### Interview Questions on This Topic
- **Q: What is Whole-Stage Code Generation?**
  - *A*: A Tungsten optimization that fuses multiple physical operators across a stage into a single Java bytecode loop, avoiding virtual function call overhead.

### What to Study Next
Open 04_join_strategies.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')